# Data Cleaning and Saving

This notebook creates the interim datasets consumed by the downstream analysis. All datasets are produced through `DataCleaner.get_clean_data`.

## Workflow

1. Load raw activity, vehicle, and user data (`registered_date` for tenure features).
2. Define the shared segmentation and churn parameters.
3. Create raw datasets for exploratory analysis (no churn / cutoff filtering).
4. Create filtered, model-ready datasets for survival analysis.

## Outputs

- **All users — raw:** merged data with no user-type segmentation, cutoff filtering, or inactivity filtering.
- **Personal users — raw:** merged and segmented personal-use data with no cutoff or inactivity filtering.
- **Professional users — raw:** merged and segmented professional-use data with no cutoff or inactivity filtering.
- **Personal users — filtered:** cutoff-filtered data with incomplete vehicle metadata removed and activity truncated at the first churn event (`churn_triggered` kept on the trigger row).
- **Professional users — filtered:** the equivalent model-ready dataset using the professional churn threshold.

The raw datasets support distributional exploration in notebook `01`. The filtered datasets are the input to notebook `04`.

**Note on dates:** feature intervals in `04` / `DataProcessor` run from each user's first to **last activity date** (empty mid-history intervals are kept; the post-churn quiet stretch is not). `churn_adjusted_date` on churn rows is `activity_date + threshold` and records definitional churn time — it does **not** extend the modelling interval grid.

## Setup and shared configuration

In [2]:
import os
import pandas as pd
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent, parent / "Coding")
    if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir()
)
os.chdir(PROJECT_ROOT)
PROJECT_ROOT

PosixPath('/Users/tomas.p/Desktop/Survival_Analysis_Thesis/Coding')

### Data imports and cleaning parameters


In [3]:
from src.constants import paths_to_files_and_folders as const
from src.data_cleaning import DataCleaner
from src.constants.segments import PERSONAL, PROFESSIONAL
from src.constants.cleaning import DEFAULT_HHI_THRESHOLD, DEFAULT_CAR_SHARE_ABS, DEFAULT_CAR_SHARE_FRACTION

In [4]:
activity_df = pd.read_csv(const.PATH_TO_RAW_ACTIVITY_DATA_1000)
vehicle_df  = pd.read_csv(const.PATH_TO_RAW_VEHICLE_DATA_1000)
user_df     = pd.read_csv(const.PATH_TO_RAW_USER_DATA_1000)
data_cleaner = DataCleaner(activity_df, vehicle_df, user_df)

# Decided churn threshold (see interval / gap analysis in notebook 01)
CHURN_THRESHOLD_DAYS_PERSONAL = PERSONAL.churn_threshold_days
CHURN_THRESHOLD_DAYS_PROFESSIONAL = PROFESSIONAL.churn_threshold_days

# Shared split criteria
HHI_THRESHOLD       = DEFAULT_HHI_THRESHOLD
CAR_SHARE_ABS       = DEFAULT_CAR_SHARE_ABS
CAR_SHARE_FRACTION  = DEFAULT_CAR_SHARE_FRACTION



## 1. Raw datasets

These datasets retain the cleaned activity history without cutoff-date or inactivity filtering. They are intended for exploratory analysis.

### 1.1 All users

In [5]:
merged_all_raw = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_nan_cols=None,
    filter_by_user_type=False,
    # return_personal_use_users=True,          
    filter_early_churners=False,
    transform_vehicle_end_year_to_present=True,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "merged_all_users_raw.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626


Cleaning complete
Final rows: 560583
Final unique users: 2755

_Step 3_
Saving File to /Users/tomas.p/Desktop/Survival_Analysis_Thesis/Coding/Data/interim/merged_all_users_raw.csv


### 1.2 Personal users


In [6]:
personal_raw = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=True,          # personal
    filter_early_churners=False,
    transform_vehicle_end_year_to_present=True,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "personal_users_raw.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Rows before user type filtering: 560583
Filtering by personal users!
Rows after user type filtering: 177113


Cleaning complete
Final rows: 177113
Final unique users: 2540

_Step 4_
Saving File to /Users/tomas.p/Desktop/Survival_Analysis_Thesis/Coding/Data/interim/personal_users_raw.csv


### 1.3 Professional users


In [7]:
professional_raw = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=False,         # professional
    filter_early_churners=False,             # raw exploratories: no span filter (needs inactivity first)
    transform_vehicle_end_year_to_present=True,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "professional_users_raw.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Rows before user type filtering: 560583
Filtering by professional users!
Rows after user type filtering: 64062


Cleaning complete
Final rows: 64062
Final unique users: 215

_Step 4_
Saving File to /Users/tomas.p/Desktop/Survival_Analysis_Thesis/Coding/Data/interim/professional_users_raw.csv


## 2. Filtered datasets

These model-ready datasets add cutoff-date filtering, remove incomplete vehicle metadata, label churn, and truncate each user's activity at the first churn event (trigger row kept; later rows dropped). Early-churners (activity span &lt; one full interval) are removed so every retained user can support at least one complete feature window.

### 2.1 Personal users


In [8]:
personal_filtered = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_inactivity=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=True,          # personal
    filter_early_churners=True,
    filter_by_set_cutoff_date=True,
    transform_vehicle_end_year_to_present=True,
    filter_nan_vehicle_metadata=True,
    threshold_value=CHURN_THRESHOLD_DAYS_PERSONAL,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "personal_users_filtered.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Rows before user type filtering: 560583
Filtering by personal users!
Rows after user type filtering: 177113

_Step 4_
Rows before vehicle metadata filtering: 177113
Rows after vehicle metadata filtering: 166877
Rows removed: 10236

_Step 5_
Filtering activity after inactivity threshold: 160 days
Rows before inactivity filtering: 166877
Rows after inactivity filtering: 133569
Rows removed: 33308
Unique users before: 2539
Unique users after: 2539
Churn-triggering rows: 1477

_Step 6_
Rows before set cutoff date filtering: 133569
Rows after set cutoff date filter

### 2.2 Professional users


In [9]:
professional_filtered = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_inactivity=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=False,         # professional
    filter_early_churners=True,
    filter_by_set_cutoff_date=True,
    transform_vehicle_end_year_to_present=True,
    filter_nan_vehicle_metadata=True,
    threshold_value=CHURN_THRESHOLD_DAYS_PROFESSIONAL, 
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "professional_users_filtered.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Rows before user type filtering: 560583
Filtering by professional users!
Rows after user type filtering: 64062

_Step 4_
Rows before vehicle metadata filtering: 64062
Rows after vehicle metadata filtering: 61148
Rows removed: 2914

_Step 5_
Filtering activity after inactivity threshold: 80 days
Rows before inactivity filtering: 61148
Rows after inactivity filtering: 36477
Rows removed: 24671
Unique users before: 215
Unique users after: 215
Churn-triggering rows: 125

_Step 6_
Rows before set cutoff date filtering: 36477
Rows after set cutoff date filtering: 36